# Wine Quality Prediction

**Objective:** Train and compare multiple classification models (Random Forest, SGD, SVC)
to predict wine quality score based on physicochemical properties.

**Dataset:** Download **"Wine Quality"** dataset from Kaggle or the UCI Machine Learning
Repository (archive.ics.uci.edu/dataset/186/wine+quality). Save as `winequality-red.csv`
in the same folder as this notebook (semicolon-separated, as distributed by UCI).

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style('whitegrid')
%matplotlib inline

## 2. Load Data & Inspect Structure

In [ ]:
df = pd.read_csv('winequality-red.csv', sep=';')
print(df.shape)
df.head()

In [ ]:
df['quality'].value_counts().sort_index()

In [ ]:
plt.figure(figsize=(7,4))
sns.countplot(x='quality', data=df)
plt.title('Distribution of Quality Scores')
plt.show()

**Class imbalance discussion:** Note here which quality scores are underrepresented
(usually scores at the extremes, like 3 and 8, have very few samples). This imbalance
means a model can achieve high accuracy just by predicting the majority classes (5 and 6)
and still perform poorly on rare classes — accuracy alone can be misleading.

## 3. EDA: Feature Distributions & Correlation

In [ ]:
df.hist(figsize=(14,10), bins=20)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 4. Feature Engineering: Binning Quality into Classes

**Justification:** Raw quality scores (3-8) create a sparse multi-class problem with
severe imbalance. Binning into **3 classes — low (3-4), medium (5-6), high (7-8)** —
gives the models a more learnable, balanced target while still being useful for a
real-world quality-screening application.

In [ ]:
def bin_quality(q):
    if q <= 4:
        return 'low'
    elif q <= 6:
        return 'medium'
    else:
        return 'high'

df['quality_class'] = df['quality'].apply(bin_quality)
df['quality_class'].value_counts()

## 5. Train/Test Split (Stratified)

In [ ]:
X = df.drop(columns=['quality', 'quality_class'])
y = df['quality_class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True))

## 6. Train 3 Classifiers

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

sgd = SGDClassifier(random_state=42, max_iter=1000)
sgd.fit(X_train_scaled, y_train)

svc = SVC(kernel='rbf', random_state=42)
svc.fit(X_train_scaled, y_train)

models = {'Random Forest': (rf, X_test), 'SGD': (sgd, X_test_scaled), 'SVC': (svc, X_test_scaled)}

## 7. Evaluate Each Model

In [ ]:
results = {}
for name, (m, X_te) in models.items():
    preds = m.predict(X_te)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    print(f"===== {name} =====")
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, preds))
    print()

## 8. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18,5))
labels = ['low', 'medium', 'high']

for ax, (name, (m, X_te)) in zip(axes, models.items()):
    preds = m.predict(X_te)
    cm = confusion_matrix(y_test, preds, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.show()

## 9. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8,5))
importances.plot(kind='barh')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.show()

importances

## 10. Comparison Table

In [ ]:
comparison = pd.DataFrame({'Model': list(results.keys()), 'Accuracy': list(results.values())})
comparison = comparison.sort_values('Accuracy', ascending=False).reset_index(drop=True)
comparison

## Conclusion

Write 2-3 sentences here identifying:
- Which model performed best on accuracy and macro F1-score (check the classification
  reports above, since accuracy alone can hide poor performance on the "high"/"low" classes)
- Which chemical features mattered most according to the Random Forest importances
- Which model you'd recommend deploying and why (consider both performance and
  interpretability)